In [1]:
!pip install datasets matplotlib numpy seaborn spacy tqdm

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 37.5 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 58.0 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.9/33.9 MB 78.4 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.3/869.3 kB 8.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 21.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 12.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 35.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 53.8 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 19.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 26.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 5.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 2

In [2]:
import pandas as pd
import numpy as np
from datasets import Dataset, load_dataset, load_from_disk
import os
import spacy
from tqdm import tqdm

In [3]:
# Load dataset
dataset_large = load_dataset("JohanHeinsen/ENO", split='train')

df = dataset_large.to_pandas()
df.shape

README.md: 0.00B [00:00, ?B/s]

ENO.csv:   0%|          | 0.00/3.40G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4898084 [00:00<?, ? examples/s]

(4898084, 5)

In [4]:
df.shape

(4898084, 5)

In [18]:
df.columns

Index(['text', 'date', 'id', 'pwa', 'newspaper'], dtype='object')

In [5]:
# Load spaCy Danish model
#nlp = spacy.load("da_core_news_sm")

# Use tqdm to track progress (optional)
tqdm.pandas()

# QUICK approach
df['article_length'] = df['text'].str.split().str.len()

# SLOW approach: Add a column with token counts (excluding punctuation and spaces)
#df['article_length'] = df['text'].progress_apply(
#    lambda text: len([token for token in nlp(text) if not token.is_punct and not token.is_space])
#)

df['date'] = pd.to_datetime(df['date'], errors='coerce', format='mixed')

In [6]:
# Group by newspaper
grouped = df.groupby('newspaper').agg(
    Period_start=('date', lambda x: x.min().year),
    Period_end=('date', lambda x: x.max().year),
    Editions=('date', lambda x: x.dt.date.nunique()),
    Articles=('text', 'count'),
    Words=('article_length', 'sum')
).reset_index()

# Create Period column
grouped['Period'] = grouped['Period_start'].astype(str) + '–' + grouped['Period_end'].astype(str)

# Reorder columns
grouped = grouped[['newspaper', 'Period', 'Editions', 'Articles', 'Words']]

# Optional: Rename columns
grouped.columns = ['Newspaper', 'Period', 'Editions', 'Articles', 'Words']

In [9]:
grouped.sum()

Newspaper    Aalborg StiftstidendeAalborgs Stifts Adresseav...
Period       1827–18461818–18271794–18471779–17821842–18441...
Editions                                                 93908
Articles                                               4898084
Words                                                473136881
dtype: object

In [7]:
df.groupby('newspaper')['text'].count()

newspaper
Aalborg Stiftstidende                             203072
Aalborgs Stifts Adresseavis                        62540
Aarhuus Stifts-Tidende                            365738
Adresseavis for Børn                                1471
Almuevennen                                         2020
Berlingske Tidende                                792663
Corsaren                                            2832
Danske Mercurius                                    2308
Den Nord-Cimbriske Tilskuer                        94609
Den Vest-Sjællandske Avis                         132439
Efterretninger fra Adresse-Contoiret i Bergen      86338
Extraordinaire Maanedlige Relationer                7372
Extraordinaire Relationer                           6887
Jyllandsposten                                      3471
Jyske Efterretninger                              150227
Kiøbenhavns Extraordinaire Relation                 2796
Kiøbenhavns Maanetlige Postrytter                   4172
Kiøbenhavns Postrytte

In [8]:
latex_table = grouped.to_latex(index=False, column_format="llrrr", escape=False)
print(latex_table)

\begin{tabular}{llrrr}
\toprule
Newspaper & Period & Editions & Articles & Words \\
\midrule
Aalborg Stiftstidende & 1827–1846 & 4519 & 203072 & 19026204 \\
Aalborgs Stifts Adresseavis & 1818–1827 & 1591 & 62540 & 5212682 \\
Aarhuus Stifts-Tidende & 1794–1847 & 8710 & 365738 & 30492484 \\
Adresseavis for Børn & 1779–1782 & 206 & 1471 & 504229 \\
Almuevennen & 1842–1844 & 161 & 2020 & 708770 \\
Berlingske Tidende & 1749–1836 & 10946 & 792663 & 113926106 \\
Corsaren & 1840–1844 & 211 & 2832 & 866368 \\
Danske Mercurius & 1677–1691 & 144 & 2308 & 256423 \\
Den Nord-Cimbriske Tilskuer & 1824–1848 & 3238 & 94609 & 9522907 \\
Den Vest-Sjællandske Avis & 1815–1842 & 3621 & 132439 & 16824428 \\
Efterretninger fra Adresse-Contoiret i Bergen & 1765–1814 & 2175 & 86338 & 5740796 \\
Extraordinaire Maanedlige Relationer & 1677–1698 & 250 & 7372 & 876360 \\
Extraordinaire Relationer & 1720–1748 & 338 & 6887 & 714399 \\
Jyllandsposten & 1838–1841 & 250 & 3471 & 933957 \\
Jyske Efterretninger & 1767–1